# Phase 5 — API Deployment

**Goal:** Deploy trained models as a FastAPI service accessible to hospital systems and dashboards.

## Contents
1. API Source Code Overview
2. Schema Validation
3. Sample Requests & Responses
4. Docker Deployment
5. Monitoring Integration

> **Note:** Full API source code lives in `phase5_deployment/main.py`. This notebook demonstrates usage and testing.

In [ ]:
import json, joblib, hashlib, pandas as pd, numpy as np
from datetime import datetime
import os
os.chdir('..') if 'notebooks' in os.getcwd() else None

print('Loading models...')
model_a = joblib.load('phase3_models/model_a_risk_rf_tuned.joblib')
model_b = joblib.load('phase3_models/model_b_claim_rf_tuned.joblib')
with open('phase3_models/feature_schema.json') as f:
    schema = json.load(f)
print('✅ Models loaded')

## 2. Schema Validation

All API inputs are validated against the feature schema before reaching the model. Invalid inputs return HTTP 422 with a descriptive error message.

In [ ]:
# Simulate Pydantic-style validation
def validate_risk_input(data: dict) -> dict:
    """Validate and sanitise risk prediction input."""
    errors = []
    if not (0 <= data.get('age', -1) <= 130): errors.append('age must be 0-130')
    if data.get('gender') not in ['M','F','Other']: errors.append('gender must be M/F/Other')
    if data.get('chronic_flag') not in [0,1]: errors.append('chronic_flag must be 0 or 1')
    if data.get('length_of_stay_hours', -1) < 0: errors.append('length_of_stay_hours must be >= 0')
    if not (1 <= data.get('visit_month',0) <= 12): errors.append('visit_month must be 1-12')
    if errors:
        raise ValueError(f'Validation failed: {errors}')
    return data

# Valid input
try:
    sample_valid = {'age':58,'gender':'M','city':'Hyderabad','chronic_flag':1,
                    'department':'Cardiology','visit_type':'ER','length_of_stay_hours':24.5,
                    'visit_month':6,'visit_dayofweek':2,'is_weekend':0,'patient_visit_freq':4,
                    'patient_avg_los':18.3,'dept_avg_los':20.1,'los_vs_dept_avg':1.22,
                    'days_since_registration':365}
    validate_risk_input(sample_valid)
    print('✅ Valid input passed validation')
except ValueError as e:
    print(f'❌ {e}')

# Invalid input
try:
    validate_risk_input({'age':200,'gender':'X','chronic_flag':5,'length_of_stay_hours':-1,'visit_month':15})
except ValueError as e:
    print(f'Expected validation error: {e}')

## 3. Sample Predictions

In [ ]:
FEATURES_A = schema['model_a_risk_features']
FEATURES_B = schema['model_b_claim_features']

def predict_risk(input_data: dict) -> dict:
    X = pd.DataFrame([{f: input_data.get(f, np.nan) for f in FEATURES_A}])
    pred = model_a.predict(X)[0]
    proba = model_a.predict_proba(X)[0]
    classes = list(model_a.classes_)
    pid = hashlib.sha256(json.dumps(input_data, sort_keys=True).encode()).hexdigest()[:16]
    return {'visit_risk': pred, 'confidence': dict(zip(classes, proba.round(4))),
            'model_version': '1.0.0', 'prediction_id': pid,
            'timestamp': datetime.utcnow().isoformat()}

def predict_claim(input_data: dict) -> dict:
    X = pd.DataFrame([{f: input_data.get(f, np.nan) for f in FEATURES_B}])
    pred = model_b.predict(X)[0]
    proba = model_b.predict_proba(X)[0]
    classes = list(model_b.classes_)
    pid = hashlib.sha256(json.dumps(input_data, sort_keys=True).encode()).hexdigest()[:16]
    revenue_risk = (pred == 'Rejected' and input_data.get('billed_amount',0) > 20000)
    return {'claim_outcome': pred, 'confidence': dict(zip(classes, proba.round(4))),
            'model_version': '1.0.0', 'prediction_id': pid,
            'timestamp': datetime.utcnow().isoformat(), 'revenue_risk_flag': revenue_risk}

# Sample risk prediction
risk_input = {'age':58,'gender':'M','city':'Hyderabad','chronic_flag':1,'department':'Cardiology',
              'visit_type':'ER','length_of_stay_hours':24.5,'visit_month':6,'visit_dayofweek':2,
              'is_weekend':0,'patient_visit_freq':4,'patient_avg_los':18.3,'dept_avg_los':20.1,
              'los_vs_dept_avg':1.22,'days_since_registration':365}
risk_response = predict_risk(risk_input)
print('=== POST /predict/risk ===')
print(json.dumps(risk_response, indent=2))

print()

# Sample claim prediction
claim_input = {'age':45,'gender':'F','city':'Chennai','chronic_flag':0,'department':'Orthopedics',
               'visit_type':'OPD','length_of_stay_hours':6.0,'billed_amount':25000.0,
               'log_billed_amount':10.13,'insurance_provider':'HealthPlus',
               'insurer_rejection_rate':0.15,'visit_month':3,'visit_dayofweek':1,'is_weekend':0,
               'patient_visit_freq':2,'bill_per_los_hour':4166.67,'payment_days_missing':1}
claim_response = predict_claim(claim_input)
print('=== POST /predict/claim ===')
print(json.dumps(claim_response, indent=2))

## 4. Docker Deployment

```bash
# Build image
docker build -t hospital-risk-api:1.0.0 phase5_deployment/

# Run with model volume mount
docker run -d \
  --name hospital-risk-api \
  -p 8000:8000 \
  -v $(pwd)/phase3_models:/app/data_outputs \
  hospital-risk-api:1.0.0

# Health check
curl http://localhost:8000/health

# Predict
curl -X POST http://localhost:8000/predict/risk \
  -H 'Content-Type: application/json' \
  -d '{"age":58,"gender":"M","city":"Hyderabad",...}'
```

Full cURL examples and error codes are documented in `phase5_deployment/DEPLOYMENT_RUNBOOK.md`.

## 5. Prediction Logging

Every prediction is appended to an audit log:

In [ ]:
import logging
logging.basicConfig(filename='phase5_deployment/prediction_audit.log',
                    level=logging.INFO, format='%(asctime)s | %(message)s')
logger = logging.getLogger(__name__)

def log_prediction(endpoint, result, input_data):
    entry = {'endpoint': endpoint, 'prediction_id': result.get('visit_risk', result.get('claim_outcome')),
             'model_version': result['model_version'], 'timestamp': result['timestamp'],
             'input_hash': hashlib.md5(json.dumps(input_data, sort_keys=True).encode()).hexdigest()}
    logger.info(json.dumps(entry))
    return entry

log_entry = log_prediction('/predict/risk', risk_response, risk_input)
print('Audit log entry:')
print(json.dumps(log_entry, indent=2))
print('\n✅ Phase 5 Complete — API ready for deployment')